# ABC Selective Attention - Structure-Sensitive Text Counting

Kaggle Bench runner for the counting task on the structure-sensitive text slice.


In [ ]:
from dataclasses import dataclass

import kaggle_benchmarks as kbench
import pandas as pd

from kaggle.shared import abc_selective_attention_utils as utils


In [ ]:
@dataclass
class CountAnswer:
    count: int


In [ ]:
DATASET_ROOT = utils.DEFAULT_DATASET_ROOT
CSV_PATH = DATASET_ROOT / 'selective_attention/structure_sensitive/text/structure_sensitive_text_full.csv'

df = utils.load_csv(CSV_PATH)
print(CSV_PATH)
print('rows:', len(df))
print('columns:', sorted(df.columns.tolist()))


In [ ]:
base_cols = [
    'seed',
    'family',
    'attentional_basis',
    'modality',
    'dimension',
    'variant',
    'structure_type',
    'structure_depth',
    'binding_distance',
    'serialization_style',
    'num_records',
]

optional_cols = [
    col
    for col in ['target_count', 'confound_count', 'confound_type', 'line_length_noise']
    if col in df.columns
]

count_eval_df = df[base_cols + optional_cols + ['count_prompt', 'gold_count']].copy()
groupings = utils.build_text_groupings(df)

print('count_eval_df columns:', count_eval_df.columns.tolist())
print('groupings:', groupings)


In [ ]:
@kbench.task(store_task=False)
def single_structure_sensitive_text_count_v1(
    llm,
    seed,
    family,
    attentional_basis,
    modality,
    dimension,
    variant,
    structure_type,
    structure_depth,
    binding_distance,
    serialization_style,
    num_records,
    count_prompt,
    gold_count,
    target_count=None,
    confound_count=None,
    confound_type=None,
    line_length_noise=None,
) -> dict:
    response = llm.prompt(count_prompt, schema=CountAnswer)
    pred = int(response.count)
    gold = int(gold_count)

    result = {
        'task': 'structure_sensitive_text_count_v1',
        'model': llm.model,
        'seed': int(seed),
        'family': family,
        'attentional_basis': attentional_basis,
        'modality': modality,
        'dimension': dimension,
        'variant': variant,
        'task_type': 'counting',
        'structure_type': structure_type,
        'structure_depth': structure_depth,
        'binding_distance': binding_distance,
        'serialization_style': serialization_style,
        'num_records': int(num_records),
        'gold_count': gold,
        'predicted_count': pred,
        'is_correct': pred == gold,
    }

    if target_count not in (None, ''):
        result['target_count'] = int(target_count)
    if confound_count not in (None, ''):
        result['confound_count'] = int(confound_count)
    if confound_type not in (None, ''):
        result['confound_type'] = confound_type
    if line_length_noise not in (None, ''):
        result['line_length_noise'] = int(line_length_noise)

    return result


In [ ]:
@kbench.task(store_task=False)
def structure_sensitive_text_count_dataset(llm) -> float:
    flat, summary = utils.run_dataset_single_model(
        item_task=single_structure_sensitive_text_count_v1,
        llm=llm,
        eval_df=count_eval_df,
        task_name='structure_sensitive_text_count_dataset',
        n_jobs=8,
        groupings=groupings,
    )
    print('=== failures: structure_sensitive_text_count_dataset ===')
    utils.print_failures(
        flat,
        columns=utils.available_columns(
            flat,
            [
                'seed',
                'family',
                'attentional_basis',
                'modality',
                'dimension',
                'variant',
                'structure_type',
                'structure_depth',
                'binding_distance',
                'serialization_style',
                'target_count',
                'confound_count',
                'confound_type',
                'line_length_noise',
                'num_records',
                'gold_count',
                'predicted_count',
            ],
        ),
    )
    return summary['accuracy'] * 100.0


In [ ]:
run = structure_sensitive_text_count_dataset.run(kbench.llm)
run


In [ ]:
%choose structure_sensitive_text_count_dataset
